# Phase II (AI-aided): layer-wise activity as a spike-death proxy\n\nClaim L1 and its decision rules: `claims.md` (fixed before the data). Data: `results/layers_activity.csv` from `export_layer_activity.py` (Akida simulator; Phase I showed simulator = card on all predictions). Run from `phase2/ai-aided/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd()))
from aiv2 import knn
from aiv2.stats import auroc_ci

FIG = knn.ROOT / "figures" / "phase2_ai"
COL = {"fog": "#4C78A8", "gaussian_noise": "#E45756", "contrast": "#54A24B"}
KEY = ["image_id", "corruption", "severity"]

a = pd.read_csv(Path.cwd() / "results" / "layers_activity.csv")
a = a[a.layer_name != "classifier"]  # potentials, not activations (layers.csv: activation 0)
a["share"] = a.nonzero_count / a.total_units
assert a.groupby(KEY).size().eq(14).all() and len(a) == 30_000 * 14, len(a)
LAYERS = a.drop_duplicates("layer_index").sort_values("layer_index").layer_name.tolist()
LAST = LAYERS[-1]
w = a.pivot_table(index=KEY, columns="layer_name", values="share")[LAYERS].reset_index()

f = knn.per_image(knn.load())   # disagree, cnn_conf, consensus per image (notebook 01)
f = f.merge(w, on=KEY, validate="1:1")
assert len(f) == 30_000
print(f"{len(a)} layer rows, last layer = {LAST}")


## Activity per layer and severity

In [ ]:
# Heatmap layer x severity, one column per corruption. Top: mean non-zero share.
# Bottom: change relative to severity 1 (diverging, centred on 0), the view L1 (a)-(c) is about.
m = a.groupby(["corruption", "severity", "layer_index"]).share.mean()
fig, axs = plt.subplots(2, 3, figsize=(13, 8), sharey=True)
rel = {}
for j, c in enumerate(knn.CORRUPTIONS):
    M = m[c].unstack("severity")                       # layer x severity
    rel[c] = M.div(M[1], axis=0) - 1
    im0 = axs[0, j].imshow(M, aspect="auto", cmap="viridis", vmin=0, vmax=m.max())
    axs[0, j].set(title=c, xticks=range(5), xticklabels=knn.SEVERITIES)
    axs[1, j].set(xlabel="severity", xticks=range(5), xticklabels=knn.SEVERITIES)
lim = max(abs(r.values).max() for r in rel.values())
for j, c in enumerate(knn.CORRUPTIONS):
    im1 = axs[1, j].imshow(rel[c], aspect="auto", cmap="RdBu", vmin=-lim, vmax=lim)
for ax in axs[:, 0]:
    ax.set(yticks=range(len(LAYERS)), yticklabels=LAYERS)
fig.colorbar(im0, ax=axs[0], label="mean non-zero share")
fig.colorbar(im1, ax=axs[1], label="change vs severity 1 (relative)")
fig.suptitle("Akida simulator: layer activity under corruption (2000 images per cell)")
fig.savefig(FIG / "l1_layer_heatmap.png", dpi=150, bbox_inches="tight")


## L1 (a)-(c): does activity die toward the last layers?

In [ ]:
# L1 (a)-(c), decision rules from claims.md.
rows = []
for c in knn.CORRUPTIONS:
    s1 = f[(f.corruption == c) & (f.severity == 1)].set_index("image_id")[LAST]
    s5 = f[(f.corruption == c) & (f.severity == 5)].set_index("image_id")[LAST]
    diff = (s5 - s1.reindex(s5.index)).values          # paired: same 2000 images
    rng = np.random.default_rng(0)
    bs = [rng.choice(diff, len(diff)).mean() for _ in range(1000)]
    drop = -rel[c][5].values                            # relative drop per layer, sev 1 -> 5
    drop_2_5 = -rel[c].loc[:, 2:5].values[-1].mean()    # last layer, mean over sev 2-5 (rule c)
    rows.append(dict(corruption=c, last_sev1=s1.mean(), last_sev5=s5.mean(), diff=diff.mean(),
                     lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5),
                     rel_drop_last=drop[-1], rel_drop_last_2_5=drop_2_5,
                     rho_depth=spearmanr(range(len(drop)), drop).statistic))
l1 = pd.DataFrame(rows)
l1["a_last_falls"] = l1.hi < 0
l1["b_grows_with_depth"] = l1.rho_depth > 0
print(l1.round(4).to_string(index=False))
print("(c) largest mean relative drop at last layer, sev 2-5:",
      l1.loc[l1.rel_drop_last_2_5.idxmax(), "corruption"])


## L1 (d): low activity and disagreement

In [ ]:
# L1 (d): does low last-layer activity flag disagreement inside a condition?
# Left: AUROC of -share at every layer (where in depth does disagreement show?).
# Right: the last layer with 95 % bootstrap intervals (the decision rule).
A = pd.DataFrame({(c, s): [roc_auc_score(g.disagree, -g[L]) for L in LAYERS]
                  for (c, s), g in f.groupby(["corruption", "severity"])}, index=LAYERS)
rows = []
for (c, s), g in f.groupby(["corruption", "severity"]):
    v, lo, hi = auroc_ci(g.disagree, -g[LAST])
    rows.append(dict(corruption=c, severity=s, auc_last=v, lo=lo, hi=hi,
                     dead=g[LAST].nunique() == 1))  # same for every image (usually silent): no ranking
d4 = pd.DataFrame(rows)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw=dict(width_ratios=[1.4, 1]))
lim = max(abs(A.values - .5).max(), .05)
im = ax[0].imshow(A, aspect="auto", cmap="RdBu_r", vmin=.5 - lim, vmax=.5 + lim)
ax[0].set(yticks=range(len(LAYERS)), yticklabels=LAYERS, xticks=range(A.shape[1]),
          xticklabels=[f"{c[:5]} {s}" for c, s in A.columns],
          title="AUROC of low activity for disagreement (0.5 = no signal)")
ax[0].tick_params(axis="x", rotation=90)
fig.colorbar(im, ax=ax[0])
for i, c in enumerate(knn.CORRUPTIONS):
    t = d4[d4.corruption == c]
    x = t.severity + (i - 1) * .1
    ax[1].errorbar(x, t.auc_last, [t.auc_last - t.lo, t.hi - t.auc_last], fmt="o-",
                   color=COL[c], capsize=2, label=c)
ax[1].axhline(.5, color="grey", lw=.8)
ax[1].set(xlabel="severity", ylabel="AUROC (95 % bootstrap)", xticks=knn.SEVERITIES,
          title=f"{LAST}: low activity flags disagreement?")
ax[1].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "l1_disagreement_by_layer.png", dpi=150)
d4["d_above_half"] = (d4.lo > .5) & ~d4.dead
print(d4.round(3).to_string(index=False))
print("(d) holds in", d4.d_above_half.sum(), "of 15 conditions;",
      d4.dead.sum(), "conditions have a dead last layer")


## Combined with notebook 01 (exploratory)

In [ ]:
# Combining the techniques (exploratory, no decision rule): does last-layer activity add
# anything to the two signals of notebook 01 (confidence, neighbour consensus)?
rows = []
for (c, s), g in f.groupby(["corruption", "severity"]):
    y = g.disagree.values
    cv = lambda X: roc_auc_score(y, cross_val_predict(LogisticRegression(max_iter=1000), X, y,
                                 cv=5, method="predict_proba")[:, 1])
    rows.append(dict(corruption=c, severity=s,
                     rho_act_consensus=spearmanr(g[LAST], g.consensus).statistic,
                     rho_act_conf=spearmanr(g[LAST], g.cnn_conf).statistic,
                     auc_conf_cons=cv(g[["cnn_conf", "consensus"]]),
                     auc_conf_cons_act=cv(g[["cnn_conf", "consensus", LAST]])))
comb = pd.DataFrame(rows)
comb["gain"] = comb.auc_conf_cons_act - comb.auc_conf_cons
print(comb.round(3).to_string(index=False))


## Findings

_Not written yet._